In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import SelectFromModel
import joblib
import warnings
warnings.filterwarnings('ignore')

class HousingPriceModel:
    def __init__(self):
        self.numeric_features = ['num_rooms', 'num_baths', 'square_meters', 'year_built', 
                               'floor', 'num_crimes', 'num_supermarkets', 'is_furnished', 
                               'has_pool', 'has_ac', 'accepts_pets']
        self.categorical_features = ['orientation', 'neighborhood']
        self.model = None
        self.error_std = None

    def preprocess_data(self, df, is_training=True):
        """
        Preprocess the dataset
        """
        # Create a copy to avoid modifying the original dataframe
        df = df.copy()
        
        if is_training:
            # Remove rows with negative or invalid values
            df = df[df['square_meters'] > 0]
            df = df[df['price'] > 0]
        
        # Convert door column to numeric by extracting floor number
        df['floor'] = df['door'].str.extract('(\d+)').astype(float)
        
        # Convert boolean columns to integers (0 or 1)
        bool_columns = ['is_furnished', 'has_pool', 'has_ac', 'accepts_pets']
        for col in bool_columns:
            df[col] = df[col].fillna(False).astype(int)
        
        # Handle categorical columns
        categorical_columns = ['orientation', 'neighborhood']
        for col in categorical_columns:
            df[col] = df[col].astype(str).fillna('missing')
        
        # Drop unnecessary columns
        df = df.drop(['id', 'door'], axis=1)
        
        return df

    def create_pipeline(self):
        """
        Create the model pipeline
        """
        # Create preprocessors for numeric and categorical data
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])

        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
        ])

        # Combine preprocessors
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, self.numeric_features),
                ('cat', categorical_transformer, self.categorical_features)
            ])

        # Create full pipeline
        self.model = Pipeline([
            ('preprocessor', preprocessor),
            ('feature_selector', SelectFromModel(LassoCV(cv=5))),
            ('regressor', LinearRegression())
        ])

    def train(self, X_train, y_train):
        """
        Train the model
        """
        self.model.fit(X_train, y_train)
        
        # Calculate error std for prediction intervals
        train_pred = self.model.predict(X_train)
        train_errors = y_train - train_pred
        self.error_std = np.std(train_errors)

    def predict(self, X):
        """
        Make predictions with confidence intervals
        """
        predictions = self.model.predict(X)
        
        # Calculate 95% prediction intervals
        pred_lower = predictions - 1.96 * self.error_std
        pred_upper = predictions + 1.96 * self.error_std
        
        return predictions, pred_lower, pred_upper

    def get_feature_importance(self):
        """
        Get feature importance scores
        """
        # Get feature names after preprocessing
        feature_names = self.numeric_features.copy()
        
        if self.categorical_features:
            onehot_encoder = self.model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
            categorical_feature_names = onehot_encoder.get_feature_names_out(self.categorical_features)
            feature_names.extend(categorical_feature_names)
        
        # Get feature selector and importance scores
        feature_selector = self.model.named_steps['feature_selector']
        selected_mask = feature_selector.get_support()
        importance_scores = feature_selector.estimator_.coef_
        
        # Create dictionary of feature importance for selected features
        selected_features_importance = {
            feature: abs(importance)
            for feature, selected, importance in zip(feature_names, selected_mask, importance_scores)
            if selected
        }
        
        return dict(sorted(selected_features_importance.items(), key=lambda x: abs(x[1]), reverse=True))

def main():
    # Initialize model
    housing_model = HousingPriceModel()
    
    # Load and preprocess training data
    print("Loading and preprocessing training data...")
    train_df = pd.read_csv('data/train.csv')
    train_df = housing_model.preprocess_data(train_df, is_training=True)
    
    # Split features and target
    X_train = train_df.drop('price', axis=1)
    y_train = train_df['price']
    
    # Create and train model
    print("Training model...")
    housing_model.create_pipeline()
    housing_model.train(X_train, y_train)
    
    # Get and print feature importance
    importance = housing_model.get_feature_importance()
    print("\nFeature Importance:")
    for feature, imp in importance.items():
        print(f"{feature}: {imp:.4f}")
    
    # Make predictions on training data to evaluate performance
    predictions, lower, upper = housing_model.predict(X_train)
    mse = mean_squared_error(y_train, predictions)
    r2 = r2_score(y_train, predictions)
    
    print("\nModel Performance:")
    print(f"MSE: {mse:.2f}")
    print(f"RMSE: {np.sqrt(mse):.2f}")
    print(f"R-squared: {r2:.3f}")
    
    # Save model
    joblib.dump(housing_model, 'housing_model.joblib')
    print("\nModel saved as housing_model.joblib")
    
    # Example of making predictions on new data
    print("\nExample predictions on first 5 samples:")
    sample_predictions, sample_lower, sample_upper = housing_model.predict(X_train.head())
    for i, (pred, low, high) in enumerate(zip(sample_predictions, sample_lower, sample_upper)):
        print(f"Sample {i+1}:")
        print(f"Predicted price: {pred:.2f}")
        print(f"95% confidence interval: ({low:.2f}, {high:.2f})")

if __name__ == "__main__":
    main()

<>:37: SyntaxWarning: invalid escape sequence '\d'
<>:37: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_29577/1025396492.py:37: SyntaxWarning: invalid escape sequence '\d'
  df['floor'] = df['door'].str.extract('(\d+)').astype(float)


Loading and preprocessing training data...
Training model...

Feature Importance:
square_meters: 195.2714
num_crimes: 74.8836
floor: 28.7768
num_baths: 7.6758
num_rooms: 3.4077
has_ac: 2.6321
has_pool: 2.4706
year_built: 0.5879
is_furnished: 0.5145
num_supermarkets: 0.4263

Model Performance:
MSE: 28395.92
RMSE: 168.51
R-squared: 0.616

Model saved as housing_model.joblib

Example predictions on first 5 samples:
Sample 1:
Predicted price: 1200.60
95% confidence interval: (870.32, 1530.88)
Sample 2:
Predicted price: 1271.80
95% confidence interval: (941.52, 1602.08)
Sample 3:
Predicted price: 918.52
95% confidence interval: (588.24, 1248.80)
Sample 4:
Predicted price: 933.87
95% confidence interval: (603.59, 1264.15)
Sample 5:
Predicted price: 1514.37
95% confidence interval: (1184.09, 1844.65)


In [3]:
import pandas as pd
import joblib

def make_predictions(test_file, model_file='housing_model.joblib'):
    """
    Make predictions on new data using saved model
    """
    # Load the model
    print("Loading model...")
    model = joblib.load(model_file)
    
    # Load and preprocess test data
    print("Loading and preprocessing test data...")
    test_df = pd.read_csv(test_file)
    test_df_processed = model.preprocess_data(test_df, is_training=False)
    
    # Make predictions
    print("Making predictions...")
    predictions, lower_bound, upper_bound = model.predict(test_df_processed)
    
    # Add predictions to original dataframe
    test_df['predicted_price'] = predictions
    test_df['prediction_lower'] = lower_bound
    test_df['prediction_upper'] = upper_bound
    
    # Save predictions
    output_file = 'predictions.csv'
    test_df[["id", "predicted_price"]].to_csv(output_file, index=False)
    print(f"\nPredictions saved to {output_file}")
    
    # Display summary statistics
    print("\nPrediction Summary:")
    print(f"Average predicted price: {predictions.mean():.2f}")
    #print(f"Median predicted price: {predictions.median():.2f}")
    print(f"Price range: {predictions.min():.2f} - {predictions.max():.2f}")
    
    # Display first few predictions
    print("\nFirst few predictions:")
    print(test_df[['id', 'predicted_price', 'prediction_lower', 'prediction_upper']].head())

if __name__ == "__main__":
    # Replace 'test.csv' with your test file name
    make_predictions('data/test.csv')

Loading model...
Loading and preprocessing test data...
Making predictions...

Predictions saved to predictions.csv

Prediction Summary:
Average predicted price: 1095.48
Price range: -81.54 - 1556.24

First few predictions:
     id  predicted_price  prediction_lower  prediction_upper
0  6253      1378.795925       1048.514603       1709.077247
1  4685      1051.102608        720.821286       1381.383930
2  1732      1011.432835        681.151513       1341.714157
3  4743      1341.794969       1011.513647       1672.076291
4  4522      1125.978513        795.697191       1456.259835
